In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, log_loss
from elo_utils import add_elo

train = pd.read_csv('../data/processed/train.csv', parse_dates=['date'])
val   = pd.read_csv('../data/processed/val.csv',   parse_dates=['date'])
tv = add_elo(pd.concat([train, val])).sort_values('date').reset_index(drop=True)

# new clue: each team's draw rate over its previous 10 matches (shift(1) = no peeking)
home = tv[['id', 'date', 'home_team', 'FTR']].rename(columns={'home_team': 'team'})
away = tv[['id', 'date', 'away_team', 'FTR']].rename(columns={'away_team': 'team'})
long = pd.concat([home, away]).sort_values(['team', 'date', 'id'])
long['is_draw'] = (long['FTR'] == 'D').astype(float)
long['draw_rate10'] = long.groupby('team')['is_draw'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=3).mean())
dr = long[['id', 'team', 'draw_rate10']]
tv = tv.merge(dr.rename(columns={'team': 'home_team', 'draw_rate10': 'home_draw_rate'}), on=['id', 'home_team'], how='left')
tv = tv.merge(dr.rename(columns={'team': 'away_team', 'draw_rate10': 'away_draw_rate'}), on=['id', 'away_team'], how='left')
tv['draw_rate_avg'] = (tv['home_draw_rate'] + tv['away_draw_rate']) / 2
tv['elo_abs_diff'] = tv['elo_diff'].abs()
print(tv.shape)   # must still be 2660 rows

base = pd.read_csv('../data/processed/feature_list_elo.csv')['feature'].tolist()
reduced = ['home_elo', 'away_elo', 'elo_diff', 'home_gd_last5', 'away_gd_last5', 'gd_diff',
           'home_form_last5_overall', 'away_form_last5_overall', 'form_diff']
feature_sets = {'A: current 14': base,
                'B: reduced (Elo + form)': reduced,
                'C: reduced + draw clues': reduced + ['elo_abs_diff', 'draw_rate_avg']}

def lr():
    return Pipeline([('sc', StandardScaler()),
                     ('clf', LogisticRegression(C=1, max_iter=2000, class_weight='balanced'))])
def rf():
    return RandomForestClassifier(n_estimators=400, max_depth=4, min_samples_leaf=10,
                                  class_weight='balanced', random_state=42, n_jobs=-1)
models = {'LogReg': lr,
          'RandomForest': rf,
          'LogReg calibrated': lambda: CalibratedClassifierCV(lr(), method='sigmoid', cv=TimeSeriesSplit(3)),
          'Ensemble LR+RF': lambda: VotingClassifier([('lr', lr()), ('rf', rf())], voting='soft')}

# walk-forward: train on all earlier seasons, test on the next one (only train+val seasons used)
seasons = sorted(tv['season'].unique())
eval_seasons = seasons[3:]            # 2011/12, 2012/13, 2013/14, 2014/15
rows = []
for fs_name, feats in feature_sets.items():
    for m_name, make in models.items():
        a, f, l = [], [], []
        for s in eval_seasons:
            tr, te = tv[tv['season'] < s], tv[tv['season'] == s]
            mu = tr[feats].mean()                      # fill blanks with earlier-seasons means only
            mdl = make().fit(tr[feats].fillna(mu), tr['target'])
            Xte = te[feats].fillna(mu)
            a.append(accuracy_score(te['target'], mdl.predict(Xte)))
            f.append(f1_score(te['target'], mdl.predict(Xte), average='macro'))
            l.append(log_loss(te['target'], mdl.predict_proba(Xte), labels=[0, 1, 2]))
        rows.append({'features': fs_name, 'model': m_name,
                     'acc': np.mean(a), 'macro_f1': np.mean(f), 'macro_f1_std': np.std(f),
                     'log_loss': np.mean(l)})

res = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
display(res.round(3))
print('Always-Home accuracy over the same seasons:',
      round(np.mean([(tv[tv['season'] == s]['target'] == 2).mean() for s in eval_seasons]), 3))
res.to_csv('../outputs/improvement_experiment.csv', index=False)

(2660, 28)


,features,model,acc,macro_f1,macro_f1_std,log_loss
11,C: reduced + draw clues,Ensemble LR+RF,0.501,0.474,0.017,0.996
3,A: current 14,Ensemble LR+RF,0.502,0.472,0.025,1.001
5,B: reduced (Elo + form),RandomForest,0.489,0.471,0.017,0.996
1,A: current 14,RandomForest,0.488,0.470,0.019,1.002
9,C: reduced + draw clues,RandomForest,0.486,0.470,0.028,0.996
7,B: reduced (Elo + form),Ensemble LR+RF,0.497,0.469,0.017,0.996
8,C: reduced + draw clues,LogReg,0.505,0.469,0.009,1.003
4,B: reduced (Elo + form),LogReg,0.506,0.469,0.011,1.005
0,A: current 14,LogReg,0.506,0.466,0.017,1.010
10,C: reduced + draw clues,LogReg calibrated,0.536,0.390,0.024,0.991


Always-Home accuracy over the same seasons: 0.453
